In [1]:
import json
import os
import dashscope
from dashscope.api_entities.dashscope_response import Role
dashscope.api_key = "sk-你的KEY"
from openai import OpenAI

# 编写你的函数
def get_current_weather(location, unit="摄氏度"):
    # 获取指定地点的天气
    temperature = -1
    if '大连' in location or 'Dalian' in location:
        temperature = 10
    if location=='上海':
        temperature = 36
    if location=='深圳':
        temperature = 37
    weather_info = {
        "location": location,
        "temperature": temperature,
        "unit": unit,
        "forecast": ["晴天", "微风"],
    }
    return json.dumps(weather_info)

# 封装模型响应函数
def get_response(messages):
    try:
        response = dashscope.Generation.call(
            model='qwen-turbo',
            messages=messages,
            functions=functions,
            result_format='message'
        )
        return response
    except Exception as e:
        print(f"API调用出错: {str(e)}")
        return None

# 使用function call进行QA
def run_conversation():
    query = "大连的天气怎样"
    messages=[{"role": "user", "content": query}]
    
    # 得到第一次响应
    response = get_response(messages)
    if not response or not response.output:
        print("获取响应失败")
        return None
        
    print('response=', response)
    
    message = response.output.choices[0].message
    messages.append(message)
    print('message=', message)
    
    # Step 2, 判断用户是否要call function
    if hasattr(message, 'function_call') and message.function_call:
        function_call = message.function_call
        tool_name = function_call['name']
        # Step 3, 执行function call
        arguments = json.loads(function_call['arguments'])
        print('arguments=', arguments)
        tool_response = get_current_weather(
            location=arguments.get('location'),
            unit=arguments.get('unit'),
        )
        tool_info = {"role": "function", "name": tool_name, "content": tool_response}
        print('tool_info=', tool_info)
        messages.append(tool_info)
        print('messages=', messages)
        
        #Step 4, 得到第二次响应
        response = get_response(messages)
        if not response or not response.output:
            print("获取第二次响应失败")
            return None
            
        print('response=', response)
        message = response.output.choices[0].message
        return message
    return message

functions = [
    {
      'name': 'get_current_weather',
      'description': 'Get the current weather in a given location.',
      'parameters': {
            'type': 'object',
            'properties': {
                'location': {
                    'type': 'string',
                    'description': 'The city and state, e.g. San Francisco, CA'
                },
                'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}
            },
        'required': ['location']
      }
    }
]

if __name__ == "__main__":
    result = run_conversation()
    if result:
        print("最终结果:", result)
    else:
        print("对话执行失败")

response= {"status_code": 200, "request_id": "05555ed3-0632-94c1-a3f6-f8b434ee559e", "code": "", "message": "", "output": {"text": null, "finish_reason": null, "choices": [{"finish_reason": "function_call", "message": {"role": "assistant", "content": "", "function_call": {"name": "get_current_weather", "arguments": "{\"location\": \"Dalian, Liaoning\"}"}}}]}, "usage": {"input_tokens": 206, "output_tokens": 22, "prompt_tokens_details": {"cached_tokens": 0}, "total_tokens": 228}}
message= {"role": "assistant", "content": "", "function_call": {"name": "get_current_weather", "arguments": "{\"location\": \"Dalian, Liaoning\"}"}}
arguments= {'location': 'Dalian, Liaoning'}
tool_info= {'role': 'function', 'name': 'get_current_weather', 'content': '{"location": "Dalian, Liaoning", "temperature": 10, "unit": null, "forecast": ["\\u6674\\u5929", "\\u5fae\\u98ce"]}'}
messages= [{'role': 'user', 'content': '大连的天气怎样'}, Message({'role': 'assistant', 'content': '', 'function_call': {'name': 'get_curr

In [13]:
from openai import OpenAI
import os

api_key=os.getenv('MODELSCOPE_API_KEY')
print(api_key)

client = OpenAI(
    base_url='https://api-inference.modelscope.cn/v1',
    api_key=api_key, # ModelScope Token
)

ms-1f01126d-e6af-48a3-958b-7e927a4951ed


In [29]:
# 创建模型调用
def get_weather(city):
    # 获取指定地点的天气
    temperature = -1
    if '大连' in city or 'Dalian' in city:
        temperature = 10
    if city=='上海':
        temperature = 36
    if city=='深圳':
        temperature = 37
    weather_info = {
        "location": city,
        "temperature": temperature,
        "unit": "摄氏度",
        "forecast": ["晴天", "微风"],
    }
    # return json.dumps(weather_info)
    return json.dumps(weather_info, ensure_ascii=False)


In [30]:
function_calls=[
    {
        'type': 'function',
        'function': {
            'name':"get_weather",
            'description':'获取指定城市的当前实时天气情况',
             "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {
                            "type": "string",
                            "description": "要查询的城市名称，例如：北京、上海、纽约。"
                        },
                    }
             },
            "required": ["city"]    # 声明哪些参数是必填的
        }
    }
]

In [31]:
def get_message(content,isStream=True):
    response = client.chat.completions.create(
        model='deepseek-ai/DeepSeek-V4-Flash',
        messages=content,
        tools=function_calls,          # 替代了 functions
        tool_choice='auto',      # 替代了 function_call
        stream=isStream
    )
    return response

In [32]:
query = "大连的天气怎样"
messages=[{"role": "user", "content": query}]
response=get_message(messages,False)

print('response=', response)


response= ChatCompletion(id='chatcmpl-62f441d5-2666-9ba4-b2ee-5662061df009', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_d743237247f1421eb104fc87', function=Function(arguments='{"city": "大连"}', name='get_weather'), type='function', index=0)], function_calls=None, reasoning_content='用户想知道大连的天气情况。我可以使用get_weather工具来查询大连的实时天气。'), delta={'role': None, 'content': '', 'tool_calls': None, 'function_calls': None, 'reasoning_content': ''})], created=1782306844, model='deepseek-ai/DeepSeek-V4-Flash', object='chat.completion', service_tier=None, system_fingerprint='', usage=CompletionUsage(completion_tokens=66, prompt_tokens=291, total_tokens=357, completion_tokens_details=None, prompt_tokens_details=None))


In [33]:
import json
# 1. 取出模型要调用的函数名和参数
tool_call = response.choices[0].message.tool_calls[0]
print(tool_call)
func_name = tool_call.function.name
func_args = json.loads(tool_call.function.arguments) # 解析出 {"city": "大连"}

print(f'func_name:{func_name},func_args;{func_args}')

ChatCompletionMessageFunctionToolCall(id='call_d743237247f1421eb104fc87', function=Function(arguments='{"city": "大连"}', name='get_weather'), type='function', index=0)
func_name:get_weather,func_args;{'city': '大连'}


In [34]:
# 2. 本地执行
if func_name == "get_weather":
    weather_result = get_weather(**func_args) 
    print("本地函数执行结果:", weather_result)
    # 输出: {"location": "大连", "temperature": 10, "unit": "摄氏度", "forecast": ["晴天", "微风"]}

本地函数执行结果: {"location": "大连", "temperature": 10, "unit": "摄氏度", "forecast": ["晴天", "微风"]}


In [35]:
# 3. 把之前的对话和工具返回结果拼起来
print('response.choices[0].message',response.choices[0].message)

messages.append(response.choices[0].message) # 把模型要调用工具的那个消息加进去
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,  # 必须带上这个 id，告诉模型这是哪个工具的返回值
    "content": weather_result      # 本地函数返回的 JSON 字符串
})

print('messages',messages)
# 4. 发起第二次请求（这次不需要传 tools 了，或者传了但不让它再调）
final_response = get_message(messages, isStream=False)

print('#########################')
print("最终回答:", final_response.choices[0].message.content)


response.choices[0].message ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_d743237247f1421eb104fc87', function=Function(arguments='{"city": "大连"}', name='get_weather'), type='function', index=0)], function_calls=None, reasoning_content='用户想知道大连的天气情况。我可以使用get_weather工具来查询大连的实时天气。')
messages [{'role': 'user', 'content': '大连的天气怎样'}, ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_d743237247f1421eb104fc87', function=Function(arguments='{"city": "大连"}', name='get_weather'), type='function', index=0)], function_calls=None, reasoning_content='用户想知道大连的天气情况。我可以使用get_weather工具来查询大连的实时天气。'), {'role': 'tool', 'tool_call_id': 'call_d743237247f1421eb104fc87', 'content': '{"location": "大连", "temperature": 10, "unit": "摄氏度", "forecast": ["晴天", "微风"]}'